In [ ]:
# Hosted D2L setup: fetch the exact helper module used to build this notebook.
from pathlib import Path
from urllib.request import urlretrieve
from importlib.metadata import PackageNotFoundError, version
import importlib.util, os, subprocess, sys

required = ['numpy', 'pandas', 'matplotlib', 'requests', 'scipy', 'pillow', 'regex', 'torch', 'torchvision']
imports = {'pillow': 'PIL'}
pinned = {}
fallbacks = {'torch': 'torch==2.11.0', 'torchvision': 'torchvision==0.26.0'}
device = os.environ.get("D2L_HOSTED_DEVICE", "auto").lower()
if device not in ("auto", "cpu", "gpu"):
    raise ValueError(f"Invalid D2L_HOSTED_DEVICE={device!r}")
if device == "auto":
    try:
        gpu = (Path("/dev/nvidia0").exists() or
               subprocess.run(["nvidia-smi", "-L"], capture_output=True,
                              timeout=5).returncode == 0)
    except (FileNotFoundError, subprocess.SubprocessError):
        gpu = False
else:
    gpu = device == "gpu"
if not gpu:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
    os.environ.setdefault("JAX_PLATFORMS", "cpu")
tensorflow_version = None
if 'pytorch' in ("tensorflow", "jax"):
    try:
        tensorflow_version = version("tensorflow")
    except PackageNotFoundError:
        pass
# Colab's CPU image currently carries a CUDA-enabled TensorFlow wheel. Its
# first ordinary tensor operation probes CUDA and emits an error-level cuInit
# diagnostic. JAX notebooks also use TensorFlow for data loading, so overlay
# the matching CPU build in both CPU variants. Keep the provider's
# ``tensorflow`` distribution metadata: other preinstalled Colab packages
# depend on that distribution name, while both wheels expose the same module.
if not gpu and 'pytorch' in ("tensorflow", "jax"):
    try:
        tensorflow_cpu_version = version("tensorflow-cpu")
    except PackageNotFoundError:
        tensorflow_cpu_version = None
    if (tensorflow_version is not None and
            tensorflow_cpu_version != tensorflow_version):
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--no-deps",
            f"tensorflow-cpu=={tensorflow_version}",
        ])
if "tf-keras" in fallbacks and tensorflow_version is not None:
    fallbacks["tf-keras"] = f"tf-keras=={tensorflow_version}"
missing = []
for package in required:
    if package in pinned:
        wanted, cpu_requirement, gpu_requirement, match = pinned[package]
        requirement = gpu_requirement if gpu else cpu_requirement
        try:
            installed = version(package)
        except PackageNotFoundError:
            installed = None
        actual = (installed.split("+", 1)[0]
                  if installed is not None and match == "public" else installed)
        if actual != wanted:
            missing.append(requirement)
    elif importlib.util.find_spec(imports.get(package, package)) is None:
        missing.append(fallbacks.get(package, package))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

mismatched = []
for package, (wanted, _, _, match) in pinned.items():
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    actual = (installed.split("+", 1)[0]
              if installed is not None and match == "public" else installed)
    if actual != wanted:
        mismatched.append(f"{package}={installed!r} (expected {wanted})")
if mismatched:
    raise RuntimeError("Hosted runtime setup failed: " + ", ".join(mismatched))

root = Path(".d2l-hosted") / "061a82116be00c585cab41d5b2a790e2a4f96d44"
package = root / "d2l"
package.mkdir(parents=True, exist_ok=True)
base = "https://raw.githubusercontent.com/smolix/d2l-neu/061a82116be00c585cab41d5b2a790e2a4f96d44/d2l"
for name in ('__init__.py', 'torch.py'):
    target = package / name
    if not target.exists():
        urlretrieve(f"{base}/{name}", target)
if str(root.resolve()) not in sys.path:
    sys.path.insert(0, str(root.resolve()))
pythonpath = os.environ.get("PYTHONPATH", "").split(os.pathsep)
if str(root.resolve()) not in pythonpath:
    os.environ["PYTHONPATH"] = os.pathsep.join(
        [str(root.resolve()), *[entry for entry in pythonpath if entry]]
    )


# Numerical Formats

Every number in our models so far has been stored in the 32-bit
floating-point format, and we have not asked whether a model needs that many
bits. Fewer bits per number would save memory and time. The model with 8
billion parameters that runs through this chapter needs 32 GB to store its
weights in 32 bits and 16 GB in a 16-bit format, and recent GPUs multiply
16-bit matrices at least twice as fast as 32-bit ones. Can we train in 16 bits
and obtain the same model?

Not without some care. Sixteen bits make the numbers coarser, and a training
run can then fail in two ways, neither of which produces an error message.
In the 16-bit format fp16, $300^2$ is already infinite, so a run can fill
with `inf` and NaN. In the 16-bit format bf16, adding 0.001 to a weight of 1
leaves the weight at 1, so a run can stop learning.

This section shows how to compute in 16 bits without these failures. Every
tensor has a *dtype*, the number format of its entries, and each format
trades *range*, how large and how small its numbers can be, against
*precision*, how finely it resolves them. Mixed-precision training keeps a
32-bit copy of the weights, so that small updates are not lost, and runs the
expensive matrix multiplications in 16 bits. With fp16 it also scales the
loss, so that small gradients do not round to zero.
that section counted the bytes per parameter of a training run;
this section changes the bytes per number. Its second part, on the dtype of
mixed expressions and on casting a model, is reference material and can be
read quickly. The code runs on any device. The
speed of 16-bit arithmetic needs a GPU with 16-bit matrix units, and
that section moves computation there.

In [ ]:
import math
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from d2l import torch as d2l

In [ ]:
import random

## Floating-Point Formats

### Range and Precision

Squaring 300 in the two 16-bit formats shows the two ways in which a format
can fail. A value of 300 is unremarkable in a network: an unnormalized logit
or an intermediate result of a variance computation can be that large.

In [ ]:
x = torch.tensor(300.0)
x.to(torch.float16)**2, x.to(torch.bfloat16)**2

The format fp16 (*half precision*) overflows to `inf`: its largest number
is 65504. The format bf16 (*brain floating point*) returns 90112 instead of
90000, a finite result that is already wrong in the third digit. Both
formats spend 16 bits. They merely divide them differently. A normal
floating-point number is stored as a sign bit $s$, an exponent $e$, and a
mantissa of $t$ bits that encodes a fraction $f \in [0, 1)$, and its value is

$$x = (-1)^s \cdot 2^{e} \cdot (1 + f).$$

The exponent bits set the range of $e$, and so how large and how small $|x|$
can be. The mantissa bits set the precision: between 1 and 2, adjacent
numbers are $2^{-t}$ apart, and between $2^e$ and $2^{e+1}$ they are
$2^{e-t}$ apart, so the relative spacing is the same at every scale. The
three formats of this section are:

| format | sign | exponent | mantissa |
|:-------|-----:|---------:|---------:|
| fp32   | 1    | 8        | 23       |
| bf16   | 1    | 8        | 7        |
| fp16   | 1    | 5        | 10       |

bf16 keeps the 8 exponent bits of fp32 and cuts the mantissa to 7 bits, so it
has the range of fp32 and coarse steps. fp16 keeps 10 mantissa bits and only 5
exponent bits: finer steps, much smaller range. Both outputs above can now be
checked by hand. With 5 exponent bits, $e$ runs from $-14$ to 15, so the
largest fp16 number is $2^{15} (2 - 2^{-10}) = 65504$. With 7 mantissa bits,
the bf16 numbers between $2^{16}$ and $2^{17}$ are $2^{9} = 512$ apart, and
90000 lies between $175 \cdot 512 = 89600$ and $176 \cdot 512 = 90112$; it
is rounded to the nearer of the two.

the figure draws the table as bit layouts.
that section treats the
floating-point representation itself in more detail.

![Bit layouts of the formats in the table, aligned at the sign bit, so that a longer bar means more bits: bf16 keeps the 8-bit exponent of fp32, and fp16 and bf16 split the same 16 bits in opposite ways.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/bg-float-formats.svg)

The function `finfo` reports what each bit budget gives. Three of its numbers
matter here: `max`, the largest finite value; `tiny`, the smallest *normal*
value; and `eps`, the spacing $2^{-t}$ of representable numbers just above 1.
What lies below `tiny`? There the exponent cannot shrink any further, and
the format continues with *subnormal* numbers, which are evenly spaced all
the way down to zero. In fp16 they are the multiples of $2^{-24}$, so a
value near $3 \times 10^{-5}$ still keeps 9 significant bits, while one near
$10^{-7}$ keeps a single bit.

In [ ]:
for dtype in (torch.float32, torch.bfloat16, torch.float16):
    fi = torch.finfo(dtype)
    print(f'{str(dtype):15} max {fi.max:10.3e}  tiny {fi.tiny:9.3e}'
          f'  eps {fi.eps:8.3e}')

bf16 has the `tiny` of fp32 and nearly the same `max`, because it has the same
exponent bits. Its `eps` of 0.0078 means two to three significant decimal
digits. fp16 resolves three to four digits, overflows above 65504, turns
subnormal below about $6.1 \times 10^{-5}$, and has no positive number below
$2^{-24} \approx 6.0 \times 10^{-8}$. Which of the two shortages is worse?
Activations and gradients of a network span many orders of magnitude, and
too little range turns them into `inf` or zero, while too little precision
only adds rounding error. This is why bf16 is the preferred 16-bit format
for training on hardware that supports it. fp16 remains in use on older GPUs
and for inference.

### Rounding of Small Updates

The coarse steps of bf16 matter most for the weights. An update of gradient
descent changes a weight by $\eta g$, which is often $10^{-3}$ or less of the
weight's own size. Under round-to-nearest, an increment smaller than half the
spacing of the numbers near the weight is lost entirely. We add $10^{-3}$ and
$10^{-4}$ to a weight of 1 in each format:

In [ ]:
w = torch.tensor(1.0)
for dtype in (torch.float32, torch.bfloat16, torch.float16):
    print(f'{str(dtype):14}: {float(w.to(dtype) + 1e-3):.7f}, '
          f'{float(w.to(dtype) + 1e-4):.7f}')

In fp32 both updates register. In bf16, whose numbers near 1 are
$2^{-7} \approx 0.0078$ apart, both round back to 1: the weight has not moved
at all. In fp16, with spacing $2^{-10} \approx 0.00098$, the larger update
survives, rounded to one step, and the smaller one is lost. A model whose
weights are stored in 16 bits can therefore stop learning even though its
gradients are correct. This is why training keeps the weights in fp32.

## Dtype Promotion and Model Casting

### Promotion

What dtype does an expression get when its operands disagree? We add an fp16
tensor to an fp32 tensor, to a Python float, and to an integer tensor:

In [ ]:
x16 = torch.ones(3, dtype=torch.float16)
x32 = torch.ones(3, dtype=torch.float32)
(x16 + x32).dtype, (x16 + 1.0).dtype, (x16 + torch.arange(3)).dtype

<!-- d2l:prose id=numerics-md-6 fw=mxnet, pytorch, jax -->

Two float tensors promote to the wider format. This is the one rule worth
remembering: an fp16 computation that meets a single stray fp32 tensor
continues in fp32, without a message, and needs twice the memory from then
on. Python scalars and integer tensors do not promote a float tensor, so a
literal such as `x + 1.0` keeps low-precision code in low precision.
<!-- d2l:prose id=numerics-md-7 fw=pytorch, jax -->

fp16 and bf16 together promote to fp32, since neither contains the other.
### Casting a Model

Parameters are tensors too, so a model can be stored in 16 bits. We use the
MLP of that section throughout this section: a hidden
layer of 256 ReLU units between the 784 pixels of a Fashion-MNIST image and
10 outputs, with weights drawn from $\mathcal{N}(0, 0.01^2)$ and zero biases.
Its 203,530 parameters take 814,120 bytes in fp32.

In [ ]:
def mlp():
    net = nn.Sequential(nn.Flatten(), nn.Linear(784, 256), nn.ReLU(),
                        nn.Linear(256, 10))
    for layer in net[1], net[3]:
        nn.init.normal_(layer.weight, std=0.01)
        nn.init.zeros_(layer.bias)
    return net

def param_bytes(net):
    return sum(p.numel() * p.element_size() for p in net.parameters())

net = mlp()
param_bytes(net)

Layers do not promote: they require the input to have the dtype of their
weights and raise an error otherwise. To change the dtype of a model we cast
the whole module; `net.to(dtype)`, or the shorthand `net.bfloat16()`,
converts every parameter and buffer.

In [ ]:
net = net.bfloat16()
X = torch.randn(8, 1, 28, 28)
try:
    net(X)
except RuntimeError as e:
    print(e)
print(net(X.bfloat16()).dtype, param_bytes(net))

In every framework the parameters now take half the bytes. What happens to
an fp32 input differs.

The error message shows the strictness: the fp32 input was refused rather
than converted.

For *inference*, a cast model needs half the memory for its parameters;
whether its predictions are still good enough must be checked for the model
and task. For *training* a cast model will not do. Its weights are exactly
the 16-bit weights that lost their updates in the rounding experiment; we
train it below and measure what this costs.

## Mixed-Precision Training

### Master Weights and 16-Bit Compute

We seem to be stuck: the weights need 32 bits, yet the speed lies in 16.
*Mixed-precision training* [@Micikevicius.Narang.Alben.ea.2018] gets
out of this by separating storage from computation. The parameters stay in
fp32, the *master weights*, so that every update registers. The matrix
multiplications of the forward and backward pass, which account for most of
the computation, run in a 16-bit format, and so do the activations they
produce. Reductions
such as the loss are computed in fp32. the figure shows one
step.

![One step of mixed-precision training: the fp32 master weights are cast to bf16 for the forward pass, which produces bf16 activations; the loss is computed in fp32; the backward pass produces gradients, and the optimizer applies them to the fp32 master weights. With fp16, the loss is also scaled up before the backward pass and the gradients are scaled back down before the update.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/bg-amp-loop.svg)

Inside a `torch.autocast` context, every operation consults a built-in
policy. Matrix multiplications and convolutions run in the 16-bit dtype, a
list of sensitive operations such as `cross_entropy` runs in fp32, and other
operations run in the dtype of their inputs. The parameters are not
touched; each eligible operation casts its inputs as it runs.

In [ ]:
net = mlp()
with torch.autocast('cpu', dtype=torch.bfloat16):
    Y = net(X)
Y.dtype, net[1].weight.dtype

The output is bf16 and the weights are fp32: the opposite of `net.bfloat16()`.
The first argument of `autocast` names the device type; on a GPU it is
`'cuda'`.

### Training in fp32 and in 16 Bits

Does 16-bit computation change what the model learns, and what does storing
the weights in 16 bits cost? We train the MLP three times with plain SGD on
minibatches of 256 Fashion-MNIST images: in fp32; as the cast model of the
previous subsection, with its weights stored in bf16 and no fp32 copy; and
with mixed precision. All three runs start from the same weights, drawn
with one random seed, and see the same minibatches in the same order.

The learning rate deserves a remark. We use 0.01, a tenth of the rate at
which that section trained this MLP, and therefore 1000
steps, about four passes over the training set. The rounding experiment
says why the rate matters: an update $\eta g$ is lost when it is smaller
than half the spacing of the numbers near its weight. At a rate of 0.01
this holds for about 90% of the updates of the first layer, as it does late
in a long run whose learning rate has been decayed. At 0.1 it holds only
for the smallest 40% of them, which carry about 2% of the update, and the
bf16 model would train as if nothing were wrong.

In [ ]:
data = d2l.FashionMNIST(batch_size=256)
batches = list(torch.utils.data.DataLoader(data.train, batch_size=256,
                                           drop_last=True))  # One epoch
val_batch = next(iter(torch.utils.data.DataLoader(data.val, batch_size=1024)))

In [ ]:
def train(net, mixed=False, num_steps=1000):
    dtype = next(net.parameters()).dtype  # The inputs follow the weights
    optimizer = torch.optim.SGD(net.parameters(), lr=0.01)
    losses = []
    for step in range(num_steps):
        X, y = batches[step % len(batches)]
        with torch.autocast('cpu', dtype=torch.bfloat16, enabled=mixed):
            loss = F.cross_entropy(net(X.to(dtype)).float(), y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        losses.append(loss.detach())
    Xv, yv = val_batch
    with torch.no_grad():
        accuracy = (net(Xv.to(dtype)).argmax(1) == yv).float().mean().item()
    return torch.stack(losses).tolist(), accuracy

One detail of `train` has nothing to do with precision. The loss values stay
on the device during training and are converted to Python numbers once, at
the end, because converting every step would force the program to wait for
each step to finish (that section).

`train` casts each minibatch to the dtype of the weights, so the cast model
computes in bf16 throughout, while `mixed=True` keeps fp32 weights and runs
the forward pass under autocast.

In [ ]:
def seeded_mlp(seed):
    torch.manual_seed(seed)  # The same seed gives the same initial weights
    return mlp()

seed = random.randrange(2**31)  # One random draw, shared by all runs
loss32, acc32 = train(seeded_mlp(seed))
loss_bf16, acc_bf16 = train(seeded_mlp(seed).bfloat16())
loss_mixed, acc_mixed = train(seeded_mlp(seed), mixed=True)
print(f'validation accuracy: fp32 {acc32:.3f}, bf16 weights {acc_bf16:.3f}, '
      f'mixed {acc_mixed:.3f}')
d2l.plot(list(range(1, 1001)), [loss32, loss_bf16, loss_mixed], 'step',
         'loss', legend=['fp32', 'bf16 weights', 'mixed precision'])

The loss curves of the fp32 run and of mixed precision lie on top of each
other, and their validation accuracies agree to within about one
percentage point. The run with bf16 weights falls behind from the first
hundred steps on: after 1000 steps its loss is about 1.15 against 0.85, and
it classifies about 64% of the validation images correctly against about
68%. Most of its first-layer updates are rounded away, as in the rounding
experiment, while mixed precision applies the same updates to fp32 master
weights and loses none of them. None of the three models is fully trained;
the comparison needs only that the runs see the same data for the same
number of steps.

What have we gained? Not speed, in this experiment: the speed of 16-bit
arithmetic depends on matrix units that compute in 16 bits, as the tensor
cores of recent GPUs do, and on models large enough to keep them busy, and
that section measures it on a GPU.

Note, however, what mixed precision does not shrink. The master weights and
the optimizer state remain fp32, so the bytes per parameter counted in
that section stay exactly as they were. The 16 GB of the opening
are what the 8-billion-parameter model occupies once it is cast for
inference. To train it, mixed precision saves only the memory of the
activations, which are stored in 16 bits, and the time of the matrix
multiplications.


### Loss Scaling for fp16

With bf16, the recipe above is complete. fp16 fails at the other end of its
range as well: small gradients *underflow*. Below $2^{-14} \approx 6.1
\times 10^{-5}$ they are stored as subnormals and lose significant bits, and
below about $3 \times 10^{-8}$, half the smallest subnormal, they are rounded
to zero. A gradient of $10^{-8}$ vanishes, but the same gradient multiplied
by $2^{16}$ is stored without trouble:

In [ ]:
g = torch.tensor(1e-8)
g.half(), (g * 2**16).half()

How many real gradients are that small? We compute the gradient of the loss
with respect to the first-layer weights of the MLP, on the first minibatch,
in fp32, and count the nonzero entries that fp16 could store only as
subnormals or zeros, with and without a factor of $2^{16}$:

In [ ]:
net = mlp()
X, y = batches[0]
F.cross_entropy(net(X), y).backward()
g = net[1].weight.grad.flatten()
g = g[g != 0]
for scale in (1, 2**16):
    print(f'scale {scale:5}: '
          f'{((g * scale).abs() < 2**-14).float().mean():.1%} subnormal, '
          f'{((g * scale).half() == 0).float().mean():.2%} zero')

About a third of the nonzero first-layer gradients lie in the subnormal
range of fp16, and about two in a thousand would round to zero. For this
small network the damage is slight, since the entries that vanish are the
smallest ones. We should be candid here: we found no small example in which
a run fails in fp16 and the remedy below rescues it. Real damage needs
nearly all of a gradient below the threshold, which can happen when a loss
is averaged over many elements or a gradient has passed through many
layers. @Micikevicius.Narang.Alben.ea.2018 report that many
networks, their ImageNet classifiers among them, trained in fp16 without
any remedy, while some did not: an object detector diverged, and it and a
large recurrent language model matched their fp32 accuracy only with the
remedy that follows.

Multiplied by $2^{16}$, none of our gradients is subnormal, and none comes
near the overflow threshold of 65504 either. This is the idea of *loss
scaling*. We multiply the loss by a large constant $S$ before the backward
pass, and because differentiation is linear, every gradient is multiplied
by $S$ as well. Before the update, we divide the gradients by $S$ in fp32,
where the small values are representable. The update is unchanged; only the
gradients in between are kept in a range that fp16 can store.

A fixed $S$ that is too large makes large gradients overflow to `inf`. The
standard remedy chooses $S$ dynamically: start high, and whenever the scaled
gradients contain `inf` or NaN, skip that update and halve $S$; after a run
of clean steps, double it again.

`torch.amp.GradScaler` implements dynamic loss scaling. The loop below trains
the MLP for 20 steps with fp16 autocast:

In [ ]:
net = mlp()
optimizer = torch.optim.SGD(net.parameters(), lr=0.1)
scaler = torch.amp.GradScaler('cpu')
for X, y in batches[:20]:
    with torch.autocast('cpu', dtype=torch.float16):
        loss = F.cross_entropy(net(X), y)
    optimizer.zero_grad()
    scaler.scale(loss).backward()  # Gradients of S * loss
    scaler.step(optimizer)         # Divide by S, skip the step on inf/NaN
    scaler.update()                # Adjust S
print(f'loss {loss.item():.3f}, loss scale {scaler.get_scale():.0f}')

The scale starts at $2^{16}$ and stays there, because no step overflowed.

On hardware that supports bf16, which includes recent GPUs, TPUs, and recent
CPUs, the standard choice is therefore bf16 mixed precision without loss
scaling. fp16 with loss scaling remains for older GPUs.

A last shortcut changes fp32 computation without changing any dtype. On
NVIDIA GPUs since the Ampere generation, an fp32 matrix multiplication can
run in *tf32* mode on the 16-bit matrix units: its inputs are rounded to the
10-bit mantissa of fp16, keeping the 8-bit exponent of fp32, and the products
are summed in fp32, so the result is faster and typically agrees with full
fp32 to three or four significant digits.

PyTorch computes fp32 matrix multiplications in full fp32 by default;
`torch.set_float32_matmul_precision('high')`, which the imports of
that section set, allows tf32.

## Numerical Failure Modes

Three failures account for most numerical trouble in 16-bit code: an overflow
that later appears as NaN, an exponential that overflows before the result
does, and a long sum that loses its small terms.

### From Overflow to NaN

A NaN in the loss is often the second symptom. The first is an overflow to
`inf`, and arithmetic on infinite values produces NaN:

In [ ]:
s = torch.tensor(60000., dtype=torch.float16) * 2  # Overflows
s, s - s

The overflow can happen many operations before the NaN reaches the loss, and
once an update contains NaN, every later result does too. When a loss turns
NaN, check ranges first: does any computation run in fp16, and do
intermediate values approach its maximum of 65504? Only then look for other
causes, such as a learning rate that is too large.

Under autocast with `GradScaler`, the check for `inf` and NaN in the
gradients catches an overflow in the step where it occurs and skips the
update.

### Log-Sum-Exp

The cross-entropy of a softmax output contains the logarithm of a sum of
exponentials of the logits. The direct evaluation of
$\log \sum_i \exp(x_i)$ overflows long before its result does:

In [ ]:
x = torch.tensor([12.0, 11.0, 10.0], dtype=torch.float16)
x.exp().sum().log(), torch.logsumexp(x, dim=0)

The result is about 12.4, but the intermediate value $e^{12} \approx 163{,}000$
exceeds 65504. The remedy is the one we used for the softmax in
that section: subtract the largest
entry $m$ before exponentiating and add it back afterwards,

$$\log \sum_i \exp(x_i) = m + \log \sum_i \exp(x_i - m).$$

No exponent on the right is positive, so nothing overflows, and the largest
term equals 1, so the sum cannot underflow to zero either. Nobody needs to
write the shift by hand: the log-softmax and cross-entropy functions of
every framework apply it.

So does `torch.logsumexp`, which the cell calls.

### Accumulation in fp32

A long sum in a 16-bit format drifts: once the running total is large, each
small increment falls below half the spacing of the numbers near the total
and is rounded away. We add $2^{-10}$, which fp16 stores exactly, 4096 times
from left to right, once into an fp16 running total and once into an fp32
running total:

In [ ]:
total16, total32 = np.float16(0), np.float32(0)
term = np.float16(2**-10)
for _ in range(4096):
    total16 += term  # fp16 accumulator
    total32 += term  # fp32 accumulator, same fp16 terms
total16, total32

The exact sum is 4, and the fp32 total reaches it. The fp16 total stops at
2, after 2048 additions: between 2 and 4 the fp16 numbers are $2^{-9}$ apart,
so $2 + 2^{-10}$ lies exactly halfway between two of them, and
round-to-nearest-even returns 2, whose last mantissa bit is even. Means over
large batches, loss totals over an epoch, and variances all follow this
pattern, which is why mixed-precision recipes compute reductions and
normalization layers in fp32. When you write a reduction over 16-bit data
yourself, give it an fp32 accumulator.

These failures share one cause: a 16-bit format has too little range or too
little precision for one intermediate value, even when the inputs and the
result are representable. Mixed precision also changes results in their last
bits by design, so a run with tf32 or mixed precision never matches an fp32
run bit for bit. Which runs can be expected to agree exactly, and how to make
them agree, is the subject of that section.

## Summary

A floating-point format divides its bits between the exponent, which sets
its range, and the mantissa, which sets its precision. Of the two 16-bit
formats, fp16 has more precision and much less range, while bf16 has the
range of fp32 and only two to three significant digits. Stored in 16 bits,
weights lose updates that are small relative to their size, so mixed-precision
training keeps fp32 master weights and runs the matrix multiplications and
activations in 16 bits; on the MLP, its loss curve matches that of fp32
training, while the same model with its weights stored in bf16 falls behind.
fp16 additionally needs loss scaling, which multiplies the loss and divides
the gradients by the same factor so that small gradients do not underflow;
bf16 does not. Compute exponentials in their shifted form and long
sums in an fp32 accumulator.

In PyTorch, `net.bfloat16()` casts a model, `torch.autocast` runs eligible
operations in 16 bits over fp32 parameters, and `torch.amp.GradScaler` scales
the loss for fp16.

## Exercises

1. **Rounded updates.** A weight $w = 1$ receives 1000 updates of
   $+10^{-3}$ each and is rounded to its storage dtype after every update.
    1. Using the `eps` values of this section and round-to-nearest, predict
       the final value of $w$ to three decimal places when it is stored in
       fp32, in fp16, and in bf16.
    1. Now the updates go to an fp32 master copy, and a bf16 working copy
       is rounded from the master copy after every update. Predict the
       final values of both copies.

1. **Mixed-precision memory.** Training with Adam in fp32 stores 16 bytes per
   parameter (that section). Suppose that `train` used Adam
   instead of SGD.
    1. In which dtype does the mixed-precision recipe of your framework store
       the weights, the gradients, and Adam's running averages between
       steps? Does mixed precision reduce the bytes per parameter?
    1. Count as stored activations the input of each linear layer and the
       logits. For the MLP of `train`, convert the per-example count to
       bytes in fp32 and in the 16-bit format `train` uses. At which batch
       size does the activation saving equal the persistent training state
       of this MLP?

1. **Loss scaling on overflow.** Under dynamic loss
   scaling (that section), a step whose unscaled gradients contain
   an `inf` or NaN is skipped and the scale is halved. After $N$
   consecutive clean steps, the scale is doubled.
    1. A skipped step must leave Adam's running averages untouched. Explain
       what would go wrong if the step instead updated them with the
       non-finite gradient, and what would go wrong if it updated them with
       the non-finite entries replaced by zeros.
    1. Explain why the scale must grow again after a run of clean steps
       instead of only shrinking.
    1. Suppose that every step overflows when the scale is above a fixed
       threshold and that no step overflows below it. Once the scale has
       settled, what fraction of steps is skipped as a function of $N$?
       Without the assumption of a fixed threshold, what does a larger $N$
       cost?

1. [code] **Normalization in fp16.** Mixed-precision recipes run
   normalization layers in fp32. To see why, consider RMSNorm with its
   gain set to one (that section implements it as a
   module). It rescales an input row $\mathbf{x}$ of width $d$ by its root
   mean square,

   $$\frac{\mathbf{x}}{\sqrt{\frac{1}{d} \sum_{i=1}^{d} x_i^2 + \epsilon}},$$

   where the small constant $\epsilon$ guards against division by zero.
   Compute it entirely in fp16 for a batch of 64 rows of width $d = 1024$
   whose entries have a standard deviation of about 100.
    1. Which intermediate quantity is the first to become `inf`, and what
       does the layer return for the affected rows?
    1. Take the squares as exact. At what input standard deviation does
       their mean exceed the fp16 maximum, and at what standard deviation
       does their sum over the $d$ entries exceed it?

5. [code] **The learning rate at which bf16 weights fail.** Give `train` a
   learning-rate argument and repeat the fp32 run and the run with bf16
   weights at the rates 0.1, 0.03, 0.01, and 0.003, with the number of
   steps chosen so that the rate times the number of steps stays at 10.
    1. At which rate does the run with bf16 weights begin to fall behind?
    1. A first-layer weight of this MLP has a size of about 0.01. What is
       the bf16 spacing near it, and how small must a gradient entry be at
       each rate for its update to be lost?

6. [code] **The e4m3fn format.** ● Print `torch.finfo(torch.float8_e4m3fn)`
   and compare its `max`, `tiny` and `eps` with those of fp16 and bf16.
   Explain the name `e4m3fn`. A format with a 4-bit exponent and a 3-bit
   mantissa that followed the IEEE convention would reserve the all-ones
   exponent for `inf` and NaN. Compute that format's largest finite value,
   and explain how `e4m3fn` reaches a `max` of 448 instead.

[Discussions](https://d2l.discourse.group/t/36756)